Concurrent Patterns notes

implementation of pydantic as well

1. asyncio.gather
2. batching - don't fire 1000 at once
do 100 times with batch of 10 at once. it's just like calling gather using for loop have 10 items at a time.
like range(0,10, step) -> [0,2,6,8] where step = 2
3. retry with exponential backoff
agent can call multiple tools, so the agent decides to call tool1 that may be an API call to get the weather.
but due to some reason that call fails with 503 or timeout or may be rate limit. they are temporary in nature. 
fix for this is to try again in some time.

Now we'll do 
model -> call -> retry -> batch -> logging with pydantic model

We'll build **Operational Baseline** now

In [ ]:
from pydantic import BaseModel
from openai import AsyncOpenAI
from dotenv import load_dotenv

load_dotenv()

def Question(BaseModel):
    question : str

def Answer(BaseModel):
    answer : str
    confidence: float
    cost: float

In [5]:
async_openai_client = AsyncOpenAI() # this gives the kitchen the power to work concurrently. 

async def ask_llm(question : Question) -> Answer:
    response = await async_openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role":"system", "content":"You're concise."},
            {"role":"user", "content": question}
        ],
        temperature=0.4
    )
    return response.choices[0].message.content